In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-_0uld8wl/unsloth_af93461af294473eab1010a50ddc08d8
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-_0uld8wl/unsloth_af93461af294473eab1010a50ddc08d8
  Resolved https://github.com/unslothai/unsloth.git to commit 5bb0bc6f7336478ff22c5e5022f8f39205b9c462
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit", # Pre-quantized vision model
    load_in_4bit = True,
)

model = FastVisionModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

In [2]:
from datasets import load_dataset


print("Loading PlantVillageVQA...")
dataset = load_dataset("SyedNazmusSakib/PlantVillageVQA", split="train")


print("Columns available:", dataset.column_names)

Loading PlantVillageVQA...
Columns available: ['image']


In [4]:
from datasets import load_dataset


raw_dataset = load_dataset("SyedNazmusSakib/PlantVillageVQA", split="train[:300]")


def convert_to_conversation(sample):
    question = sample.get("question", "Diagnose the disease in this plant image.")
    answer = sample.get("answer", sample.get("label", "Healthy crop."))

    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": sample["image"]},
                    {"type": "text", "text": str(question)}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": str(answer)}
                ]
            }
        ]
    }


formatted_dataset = [convert_to_conversation(sample) for sample in raw_dataset]

print(f"Dataset successfully formatted! Total samples: {len(formatted_dataset)}")

Dataset successfully formatted! Total samples: 300


In [5]:
from trl import SFTTrainer, SFTConfig
from unsloth import UnslothVisionDataCollator

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = formatted_dataset, # Now correctly populated!
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        output_dir = "outputs",
        save_strategy = "epoch",
    ),
)

trainer_stats = trainer.train()


Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 1 | Total steps = 38
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 10,092,544 of 8,301,468,160 (0.12% trained)


Step,Training Loss
5,3.519431
10,3.251661
15,2.481701
20,1.543385
25,0.674771
30,0.175502
35,0.023161


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-38/tokenizer_config.json.


In [ ]:
import os

HF_TOKEN = os.getenv("HF_TOKEN")
HF_REPO = "DivyaSharma02/qwen2-vl-crop-adapter"


model.push_to_hub_merged(
    HF_REPO,
    tokenizer,
    save_method = "lora",
    token = HF_TOKEN
)

print("Adapter pushed to Hugging Face successfully!")

No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   1%|          | 16.0MB / 1.69GB            

No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [09:17<00:00, 139.25s/it]


Unsloth: Merge process complete. Saved to `/content/DivyaSharma02/qwen2-vl-crop-adapter`
Adapter pushed to Hugging Face successfully!


In [10]:
from unsloth import FastVisionModel
from transformers import TextStreamer


FastVisionModel.for_inference(model)


image = raw_dataset[0]["image"]
# 3. Define prompt conversation format
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Diagnose the disease in this crop leaf and suggest immediate steps."}
        ]
    }
]

# 4. Apply chat template and tokenize inputs
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt"
).to("cuda")


text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True
)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Healthy crop.<|im_end|>


In [11]:
KNOWLEDGE_BASE = [
    {
        "disease": "Leaf Blight",
        "treatment_en": "Apply Mancozeb 75% WP at 2.5g/liter of water. Spray in early morning or evening. Repeat every 10-14 days if symptoms persist. Remove and destroy infected leaves.",
        "treatment_hi": "मैंकोज़ेब 75% WP 2.5 ग्राम प्रति लीटर पानी में मिलाकर छिड़कें। सुबह या शाम के समय छिड़काव करें।",
        "chemical": "Mancozeb",
        "dosage_per_acre": "500g in 200L water",
        "cost_inr": 350,
        "avoid_if_rain_within_hours": 24,
    },
    {
        "disease": "Powdery Mildew",
        "treatment_en": "Apply Sulphur 80% WP at 2g/liter, or Hexaconazole at 1ml/liter. Ensure good air circulation between plants.",
        "treatment_hi": "सल्फर 80% WP 2 ग्राम प्रति लीटर का छिड़काव करें। पौधों के बीच अच्छी हवा का संचार सुनिश्चित करें।",
        "chemical": "Sulphur / Hexaconazole",
        "dosage_per_acre": "400g Sulphur in 200L water",
        "cost_inr": 280,
        "avoid_if_rain_within_hours": 12,
    },
    {
        "disease": "Bacterial Wilt",
        "treatment_en": "Remove and burn infected plants immediately to prevent spread. Apply Streptocycline 100ppm as soil drench. Practice crop rotation for 2-3 seasons.",
        "treatment_hi": "संक्रमित पौधों को तुरंत हटाकर जला दें। स्ट्रेप्टोसाइक्लिन 100ppm मिट्टी में डालें।",
        "chemical": "Streptocycline",
        "dosage_per_acre": "1g in 10L water for soil drench",
        "cost_inr": 450,
        "avoid_if_rain_within_hours": 6,
    },
    {
        "disease": "Rust",
        "treatment_en": "Apply Propiconazole 25% EC at 1ml/liter of water. Spray at first sign of orange-brown pustules. Ensure full leaf coverage.",
        "treatment_hi": "प्रोपिकोनाज़ोल 25% EC 1 मिली प्रति लीटर पानी में छिड़कें।",
        "chemical": "Propiconazole",
        "dosage_per_acre": "200ml in 200L water",
        "cost_inr": 520,
        "avoid_if_rain_within_hours": 24,
    },
    {
        "disease": "Healthy",
        "treatment_en": "No disease detected. Continue regular monitoring, balanced fertilization, and proper irrigation.",
        "treatment_hi": "कोई बीमारी नहीं मिली। नियमित निगरानी जारी रखें।",
        "chemical": "None",
        "dosage_per_acre": "N/A",
        "cost_inr": 0,
        "avoid_if_rain_within_hours": 0,
    },
]

def find_treatment(disease_text, language="English"):
    for item in KNOWLEDGE_BASE:
        if item["disease"].lower() in disease_text.lower():
            result = item.copy()
            result["treatment"] = item["treatment_hi"] if language == "Hindi" else item["treatment_en"]
            return result
    return {"disease": "Unknown", "treatment": "Consult your local agriculture office.", "cost_inr": 0, "chemical": "N/A", "dosage_per_acre": "N/A", "avoid_if_rain_within_hours": 0}

print("Knowledge base ready!")

Knowledge base ready!


In [1]:
import os
import requests

# Fetch key securely from environment
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")


def check_rain(lat, lon):
  try:
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={OPENWEATHER_API_KEY}&units=metric"
    data = requests.get(url, timeout=6).json()
    next_24h = data.get("list", [])[:8]
    rain_chance = max([item.get("pop", 0) for item in next_24h], default=0)
    return rain_chance >= 0.4, round(rain_chance * 100)
  except Exception:
    return False, 0


print("Weather function ready!")

Weather function ready!


In [13]:
from PIL import Image

def diagnose(image_path):
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Diagnose the disease in this crop leaf and suggest immediate steps."}
    ]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    output_ids = model.generate(**inputs, max_new_tokens=128, use_cache=True)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Diagnosis function ready!")

Diagnosis function ready!


In [14]:
import gradio as gr

def run_app(image, lat, lon):
    if image is None:
        return "Please upload a photo."
    disease_text = diagnose(image)
    info = find_treatment(disease_text)
    will_rain, rain_pct = check_rain(lat, lon)

    result = f"**Diagnosis:** {info['disease']}\n\n**Treatment:** {info['treatment']}\n\n**Cost:** ₹{info['cost_inr']}/acre"
    if will_rain:
        result += f"\n\n⚠️ Rain likely ({rain_pct}%) in next 24h — spray may wash off."
    return result

demo = gr.Interface(
    fn=run_app,
    inputs=[
        gr.Image(type="filepath", label="Leaf Photo"),
        gr.Number(label="Latitude", value=26.9124),
        gr.Number(label="Longitude", value=75.7873),
    ],
    outputs="markdown",
    title="Crop Disease Diagnosis"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://db0b553b11c23282b7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
